## The State of Tax Justice: Estimate misalignment for 2019

- Author: Mario Cuenda García, based on Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 November 2024
- Last updated:

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [15]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.0f}'.format

## Step 1. Generate the template datasets

### Step 1.1. Generate the dataset with Unique ISO parents

In [16]:
# Open the original dataset
iso_parents = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep just the following columns: iso_parent and year
iso_parents = iso_parents[['iso_parent', 'year']]

# Keep every unique combination of iso_parent and year
iso_parents = iso_parents.drop_duplicates(subset=['iso_parent', 'year'])

# Sort by year, then iso_parent
iso_parents = iso_parents.sort_values(by=['year', 'iso_parent'])

# Filter by year
iso_parents_2016= iso_parents[iso_parents['year'] == 2016]
iso_parents_2017= iso_parents[iso_parents['year'] == 2017]
iso_parents_2018= iso_parents[iso_parents['year'] == 2018]
iso_parents_2019= iso_parents[iso_parents['year'] == 2019]
iso_parents_2020= iso_parents[iso_parents['year'] == 2020]
iso_parents_2021= iso_parents[iso_parents['year'] == 2021]

# OPTIONAL: Print the count of how many unique iso_partner values there are
print(iso_parents_2016['iso_parent'].nunique())
print(iso_parents_2017['iso_parent'].nunique())
print(iso_parents_2018['iso_parent'].nunique())
print(iso_parents_2019['iso_parent'].nunique())
print(iso_parents_2020['iso_parent'].nunique())
print(iso_parents_2021['iso_parent'].nunique())

iso_parents_2019

26
38
46
50
52
52


,iso_parent,year
153,ARG,2019
267,AUS,2019
763,AUT,2019
837,BEL,2019
1062,BMU,2019
1647,BRA,2019
1896,CAN,2019
1984,CHE,2019
2725,CHL,2019
2813,CHN,2019


### Step 1.2. Generate the dataset with unique iso_partners

In [17]:
## Download the relevant dataset
iso_partners = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
iso_partners = iso_partners[~iso_partners['iso_partner'].isin(non_countries)]

# Keep just the following columns: iso_partner and year
iso_partners = iso_partners[['iso_partner', 'year']]

# Sort by year, then iso_partner
iso_partners = iso_partners.sort_values(by=['year', 'iso_partner'])

# Keep every unique combination of iso_partner and year
iso_partners = iso_partners.drop_duplicates(subset=['iso_partner', 'year'])

# Filter by year
iso_partners_2016= iso_partners[iso_partners['year'] == 2016]
iso_partners_2017= iso_partners[iso_partners['year'] == 2017]
iso_partners_2018= iso_partners[iso_partners['year'] == 2018]
iso_partners_2019= iso_partners[iso_partners['year'] == 2019]
iso_partners_2020= iso_partners[iso_partners['year'] == 2020]
iso_partners_2021= iso_partners[iso_partners['year'] == 2021]


# Optional: Print the count of how many unique iso_partner values there are 
print(iso_partners_2016['iso_partner'].nunique())
print(iso_partners_2017['iso_partner'].nunique())
print(iso_partners_2018['iso_partner'].nunique())
print(iso_partners_2019['iso_partner'].nunique())
print(iso_partners_2020['iso_partner'].nunique())
print(iso_partners_2021['iso_partner'].nunique())

183
215
213
210
212
211


### Step 1.3. Generate the template dataset

In [18]:
# Perform a cross join to merge all values of iso_partners_ with each value of iso_parents
iso_combinations_2016 = iso_parents_2016.assign(key=1).merge(iso_partners_2016.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2017 = iso_parents_2017.assign(key=1).merge(iso_partners_2017.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2018 = iso_parents_2018.assign(key=1).merge(iso_partners_2018.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2019 = iso_parents_2019.assign(key=1).merge(iso_partners_2019.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2020 = iso_parents_2020.assign(key=1).merge(iso_partners_2020.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2021 = iso_parents_2021.assign(key=1).merge(iso_partners_2021.assign(key=1), on='key').drop('key', axis=1)

# Concatenate all the years
template_dataset = pd.concat([iso_combinations_2016, iso_combinations_2017, iso_combinations_2018, iso_combinations_2019, iso_combinations_2020, iso_combinations_2021])

# Drop year_y
template_dataset = template_dataset.drop(columns=['year_y'])
# Rename year_x to year
template_dataset = template_dataset.rename(columns={'year_x': 'year'})
# Order columns by iso_parent then iso_partner then year
template_dataset = template_dataset[['iso_parent', 'iso_partner', 'year']]
# Generate new column called cbcr_estimates
template_dataset['cbcr_estimates'] = np.nan

template_dataset

,iso_parent,iso_partner,year,cbcr_estimates
0,AUS,ABW,2016,NaN
1,AUS,AFG,2016,NaN
2,AUS,AGO,2016,NaN
3,AUS,ALB,2016,NaN
4,AUS,AND,2016,NaN
...,...,...,...,...
10967,ZAF,XKV,2021,NaN
10968,ZAF,YEM,2021,NaN
10969,ZAF,ZAF,2021,NaN
10970,ZAF,ZMB,2021,NaN


## Step 2. Define the misalignment formula

In [19]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## Step 3. Calculate misalignment for sample with full information

### Step 3.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [20]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]

### Step 3.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [21]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]

### Step 3.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


The end of this cell shows, among other things, the misaligned profits and the theoretical profits, as well as the CBCR variables.

**More importantly, if for whatever reason we wanted to just run this cell without the "bad reporters", we could just modify the cell to run like the cell from Step 5.2. in order to obtain a dataset of profit shifting without the "bad reporters". In a way, this cell is not really necessary for the rest of the notebook. It just shows an intermediary steps if we want to calculate the misalignment for the "good reporters" only.**

In [22]:
misalignment_2019 = cbcr_sample[cbcr_sample['year'] == 2019].copy()
misalignment_2019 = calculate_misalignment(misalignment_2019, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Keep only the first occurrence of these unique variables for each 'iso_partner'
unique_columns = misalignment_2019.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

# Keep iso_parent, iso_partner, year, misaligned_profit, theoretical_profit, profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
misalignment_2019 = misalignment_2019[['iso_parent', 'iso_partner', 'year', 'misaligned_profit', 'theoretical_profit', 'profit_loss_before_income_tax_corrected', 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']]

# Show if iso_partner = USA
misalignment_2019[misalignment_2019['iso_partner'] == 'USA']

/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_6423/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip
15,ARG,USA,2019,0,"-5,802,608","181,774,425",172,"360,002,944","1,709,709,185","8,737,087","1,611,489,867","380,612,773","20,609,830",0
90,AUS,USA,2019,"-1,193,161,484","5,229,936,837","3,446,318,835","90,483","45,060,586,867","35,884,418,881","4,596,266,761","142,842,000,000","64,543,875,904","19,483,289,037",21
116,BEL,USA,2019,"-25,579,169,157","26,726,269,157","1,147,100,000","60,400","38,985,700,000","15,954,000,000","3,068,140,008","65,191,100,000","49,126,500,000","10,140,800,000",22
211,BMU,USA,2019,"-370,913,937","5,942,394,792","5,261,472,357","123,442","76,520,956,341","53,230,845,962","6,270,485,743","87,178,568,755","90,819,277,063","14,298,320,722",22
250,BRA,USA,2019,"-1,892,674,949","11,571,295,652","6,652,364,941","122,779","60,343,739,039","22,248,121,267","6,236,807,319","38,439,013,015","78,134,721,511","17,790,982,472",4
263,CAN,USA,2019,"-4,253,408,506","76,653,970,506","72,400,562,000","630,580","419,710,000,000","473,721,000,000","32,031,584,872","1,100,520,000,000","511,869,000,000","92,158,921,000",NaN
414,CHE,USA,2019,"-6,102,719,181","24,187,979,255","16,795,690,714","330,522","265,678,000,000","86,578,051,774","16,789,532,644","399,132,000,000","327,422,000,000","61,743,847,053",82
435,CHL,USA,2019,"-16,105,815","340,034,526","233,934,111","7,051","5,363,699,184","2,022,810,696","358,169,788","3,394,198,472","5,581,580,603","217,881,419",0
559,CHN,USA,2019,"-7,073,548,564","13,957,386,530","4,323,736,477","85,724","101,597,000,000","59,120,703,102","4,354,523,742","70,856,997,677","129,675,000,000","28,235,223,172",85
698,CYM,USA,2019,"-4,767,313,692","7,916,057,406","-203,317,143","57,366","23,448,117,036","8,255,585,218","2,914,021,849","21,412,055,469","30,214,374,199","6,766,257,163",35


## Step 4. Generate the dataset for the "bad reporters"

For the "bad" reporters, we are going to assume that their MNEs behave like the "average" MNE in the countries that report correctly. To do that we need to:

1. First aggregate the variables reported by the CBCR by partner countries. For instance, we see that on aggregate, there are 80m employees reported.
2. Then we look at the share corresponding by partners. For instance, 18m employees are reported in the USA. how many does the USA have. In short, roughly 24% of all employees reported are in the USA.
3. We then assume that 24% of the toal employees reported by the "bad" reporters are assigned to the US. And we repeat with all of them



### Step 4.1. Generate the Total Sums of Variables, and the Total Sums by Partners.

- The first bloc of lines calculates the total of the variables, and generates a new variable (e.g. total_n_employees) in the dataset.
- The second bloc of lines groups by iso_partner and calculates the total sums by partners. (e.g. how many employees are reported by the CBCR countries, say, in the USA)
- The third bloc of lines merges the total sums by partners to the dataset

In [23]:
# Calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_before_income_tax_corrected = misalignment_2019['profit_loss_before_income_tax_corrected'].sum()
total_n_employees = misalignment_2019['n_employees'].sum()
total_unrelated_party_revenues = misalignment_2019['unrelated_party_revenues'].sum()
total_tangible_assets_except_cash = misalignment_2019['tangible_assets_except_cash'].sum()
total_payroll = misalignment_2019['payroll'].sum()
total_stated_capital = misalignment_2019['stated_capital'].sum()
total_total_revenues = misalignment_2019['total_revenues'].sum()
total_related_party_revenues = misalignment_2019['related_party_revenues'].sum()
total_holding_or_managing_ip = misalignment_2019['holding_or_managing_ip'].sum()

misalignment_2019['total_profit_loss_before_income_tax_corrected'] = total_profit_loss_before_income_tax_corrected
misalignment_2019['total_n_employees'] = total_n_employees
misalignment_2019['total_unrelated_party_revenues'] = total_unrelated_party_revenues
misalignment_2019['total_tangible_assets_except_cash'] = total_tangible_assets_except_cash
misalignment_2019['total_payroll'] = total_payroll
misalignment_2019['total_stated_capital'] = total_stated_capital
misalignment_2019['total_total_revenues'] = total_total_revenues
misalignment_2019['total_related_party_revenues'] = total_related_party_revenues
misalignment_2019['total_holding_or_managing_ip'] = total_holding_or_managing_ip

# Group by iso_partner and calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_by_partner = misalignment_2019.groupby('iso_partner')['profit_loss_before_income_tax_corrected'].sum().reset_index()
total_profit_loss_by_partner = total_profit_loss_by_partner.rename(columns={'profit_loss_before_income_tax_corrected': 'total_profit_loss_by_partner'})

total_n_employees_by_partner = misalignment_2019.groupby('iso_partner')['n_employees'].sum().reset_index()
total_n_employees_by_partner = total_n_employees_by_partner.rename(columns={'n_employees': 'total_n_employees_by_partner'})

total_unrelated_party_revenues_by_partner = misalignment_2019.groupby('iso_partner')['unrelated_party_revenues'].sum().reset_index()
total_unrelated_party_revenues_by_partner = total_unrelated_party_revenues_by_partner.rename(columns={'unrelated_party_revenues': 'total_unrelated_party_revenues_by_partner'})

total_tangible_assets_except_cash_by_partner = misalignment_2019.groupby('iso_partner')['tangible_assets_except_cash'].sum().reset_index()
total_tangible_assets_except_cash_by_partner = total_tangible_assets_except_cash_by_partner.rename(columns={'tangible_assets_except_cash': 'total_tangible_assets_except_cash_by_partner'})

total_payroll_by_partner = misalignment_2019.groupby('iso_partner')['payroll'].sum().reset_index()
total_payroll_by_partner = total_payroll_by_partner.rename(columns={'payroll': 'total_payroll_by_partner'})

total_stated_capital_by_partner = misalignment_2019.groupby('iso_partner')['stated_capital'].sum().reset_index()
total_stated_capital_by_partner = total_stated_capital_by_partner.rename(columns={'stated_capital': 'total_stated_capital_by_partner'})

total_total_revenues_by_partner = misalignment_2019.groupby('iso_partner')['total_revenues'].sum().reset_index()
total_total_revenues_by_partner = total_total_revenues_by_partner.rename(columns={'total_revenues': 'total_total_revenues_by_partner'})

total_related_party_revenues_by_partner = misalignment_2019.groupby('iso_partner')['related_party_revenues'].sum().reset_index()
total_related_party_revenues_by_partner = total_related_party_revenues_by_partner.rename(columns={'related_party_revenues': 'total_related_party_revenues_by_partner'})

total_holding_or_managing_ip_by_partner = misalignment_2019.groupby('iso_partner')['holding_or_managing_ip'].sum().reset_index()
total_holding_or_managing_ip_by_partner = total_holding_or_managing_ip_by_partner.rename(columns={'holding_or_managing_ip': 'total_holding_or_managing_ip_by_partner'})

# Merge the total profit loss by partner back into the misalignment_2019 dataframe
misalignment_2019 = misalignment_2019.merge(total_profit_loss_by_partner, on='iso_partner', how='left')
misalignment_2019 = misalignment_2019.merge(total_n_employees_by_partner, on='iso_partner', how='left')
misalignment_2019 = misalignment_2019.merge(total_unrelated_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2019 = misalignment_2019.merge(total_tangible_assets_except_cash_by_partner, on='iso_partner', how='left')
misalignment_2019 = misalignment_2019.merge(total_payroll_by_partner, on='iso_partner', how='left')
misalignment_2019 = misalignment_2019.merge(total_stated_capital_by_partner, on='iso_partner', how='left')
misalignment_2019 = misalignment_2019.merge(total_total_revenues_by_partner, on='iso_partner', how='left')
misalignment_2019 = misalignment_2019.merge(total_related_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2019 = misalignment_2019.merge(total_holding_or_managing_ip_by_partner, on='iso_partner', how='left')

misalignment_2019[misalignment_2019['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner
15,ARG,USA,2019,0,"-5,802,608","181,774,425",172,"360,002,944","1,709,709,185","8,737,087","1,611,489,867","380,612,773","20,609,830",0,"5,252,345,950,571","138,818,542","51,656,500,864,631","34,559,985,418,529","3,580,814,046,076","59,293,915,820,799","72,742,547,231,696","21,064,699,384,324","17,102","512,336,028,057","28,583,144","14,719,262,499,974","7,540,458,479,488","1,451,938,537,431","18,882,661,702,751","19,112,391,499,318","4,391,251,531,039","1,386"
90,AUS,USA,2019,"-1,193,161,484","5,229,936,837","3,446,318,835","90,483","45,060,586,867","35,884,418,881","4,596,266,761","142,842,000,000","64,543,875,904","19,483,289,037",21,"5,252,345,950,571","138,818,542","51,656,500,864,631","34,559,985,418,529","3,580,814,046,076","59,293,915,820,799","72,742,547,231,696","21,064,699,384,324","17,102","512,336,028,057","28,583,144","14,719,262,499,974","7,540,458,479,488","1,451,938,537,431","18,882,661,702,751","19,112,391,499,318","4,391,251,531,039","1,386"
116,BEL,USA,2019,"-25,579,169,157","26,726,269,157","1,147,100,000","60,400","38,985,700,000","15,954,000,000","3,068,140,008","65,191,100,000","49,126,500,000","10,140,800,000",22,"5,252,345,950,571","138,818,542","51,656,500,864,631","34,559,985,418,529","3,580,814,046,076","59,293,915,820,799","72,742,547,231,696","21,064,699,384,324","17,102","512,336,028,057","28,583,144","14,719,262,499,974","7,540,458,479,488","1,451,938,537,431","18,882,661,702,751","19,112,391,499,318","4,391,251,531,039","1,386"
211,BMU,USA,2019,"-370,913,937","5,942,394,792","5,261,472,357","123,442","76,520,956,341","53,230,845,962","6,270,485,743","87,178,568,755","90,819,277,063","14,298,320,722",22,"5,252,345,950,571","138,818,542","51,656,500,864,631","34,559,985,418,529","3,580,814,046,076","59,293,915,820,799","72,742,547,231,696","21,064,699,384,324","17,102","512,336,028,057","28,583,144","14,719,262,499,974","7,540,458,479,488","1,451,938,537,431","18,882,661,702,751","19,112,391,499,318","4,391,251,531,039","1,386"
250,BRA,USA,2019,"-1,892,674,949","11,571,295,652","6,652,364,941","122,779","60,343,739,039","22,248,121,267","6,236,807,319","38,439,013,015","78,134,721,511","17,790,982,472",4,"5,252,345,950,571","138,818,542","51,656,500,864,631","34,559,985,418,529","3,580,814,046,076","59,293,915,820,799","72,742,547,231,696","21,064,699,384,324","17,102","512,336,028,057","28,583,144","14,719,262,499,974","7,540,458,479,488","1,451,938,537,431","18,882,661,702,751","19,112,391,499,318","4,391,251,531,039","1,386"
263,CAN,USA,2019,"-4,253,408,506","76,653,970,506","72,400,562,000","630,580","419,710,000,000","473,721,000,000","32,031,584,872","1,100,520,000,000","511,869,000,000","92,158,921,000",NaN,"5,252,345,950,571","138,818,542","51,656,500,864,631","34,559,985,418,529","3,580,814,046,076","59,293,915,820,799","72,742,547,231,696","21,064,699,384,324","17,102","512,336,028,057","28,583,144","14,719,262,499,974","7,540,458,479,488","1,451,938,537,431","18,882,661,702,751","19,112,391,499,318","4,391,251,531,039","1,386"
414,CHE,USA,2019,"-6,102,719,181","24,187,979,255","16,795,690,714","330,522","265,678,000,000","86,578,051,774","16,789,532,644","399,132,000,000","327,422,000,000

### Step 4.2. Calculate the shares for all the variables

In [24]:
# Final Misalignment
final_misalignment_2019 = misalignment_2019

# Calculate the shares for all variables
final_misalignment_2019['share_reported_total_profit_loss_by_partner'] = misalignment_2019['total_profit_loss_by_partner'] / misalignment_2019['total_profit_loss_before_income_tax_corrected']
final_misalignment_2019['share_reported_total_n_employees_by_partner'] = misalignment_2019['total_n_employees_by_partner'] / misalignment_2019['total_n_employees']
final_misalignment_2019['share_reported_total_unrelated_party_revenues_by_partner'] = misalignment_2019['total_unrelated_party_revenues_by_partner'] / misalignment_2019['total_unrelated_party_revenues']
final_misalignment_2019['share_reported_total_tangible_assets_except_cash_by_partner'] = misalignment_2019['total_tangible_assets_except_cash_by_partner'] / misalignment_2019['total_tangible_assets_except_cash']
final_misalignment_2019['share_reported_total_payroll_by_partner'] = misalignment_2019['total_payroll_by_partner'] / misalignment_2019['total_payroll']
final_misalignment_2019['share_reported_total_stated_capital_by_partner'] = misalignment_2019['total_stated_capital_by_partner'] / misalignment_2019['total_stated_capital']
final_misalignment_2019['share_reported_total_total_revenues_by_partner'] = misalignment_2019['total_total_revenues_by_partner'] / misalignment_2019['total_total_revenues']
final_misalignment_2019['share_reported_total_related_party_revenues_by_partner'] = misalignment_2019['total_related_party_revenues_by_partner'] / misalignment_2019['total_related_party_revenues']
final_misalignment_2019['share_reported_total_holding_or_managing_ip_by_partner'] = misalignment_2019['total_holding_or_managing_ip_by_partner'] / misalignment_2019['total_holding_or_managing_ip']

# Give me column 'share reported' with 2 decimals
pd.options.display.float_format = '{:,.2f}'.format

final_misalignment_2019[final_misalignment_2019['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
15,ARG,USA,2019,0.00,"-5,802,608.29","181,774,425.20",172.00,"360,002,943.70","1,709,709,185.00","8,737,087.44","1,611,489,867.00","380,612,773.30","20,609,829.55",0.00,"5,252,345,950,571.46","138,818,542.00","51,656,500,864,631.46","34,559,985,418,528.91","3,580,814,046,075.57","59,293,915,820,799.34","72,742,547,231,696.05","21,064,699,384,324.01","17,102.00","512,336,028,056.76","28,583,144.00","14,719,262,499,973.70","7,540,458,479,487.50","1,451,938,537,430.88","18,882,661,702,751.00","19,112,391,499,318.30","4,391,251,531,038.75","1,386.00",0.10,0.21,0.28,0.22,0.41,0.32,0.26,0.21,0.08
90,AUS,USA,2019,"-1,193,161,483.53","5,229,936,837.11","3,446,318,835.00","90,483.00","45,060,586,867.00","35,884,418,881.00","4,596,266,760.66","142,842,000,000.00","64,543,875,904.00","19,483,289,037.00",21.00,"5,252,345,950,571.46","138,818,542.00","51,656,500,864,631.46","34,559,985,418,528.91","3,580,814,046,075.57","59,293,915,820,799.34","72,742,547,231,696.05","21,064,699,384,324.01","17,102.00","512,336,028,056.76","28,583,144.00","14,719,262,499,973.70","7,540,458,479,487.50","1,451,938,537,430.88","18,882,661,702,751.00","19,112,391,499,318.30","4,391,251,531,038.75","1,386.00",0.10,0.21,0.28,0.22,0.41,0.32,0.26,0.21,0.08
116,BEL,USA,2019,"-25,579,169,156.77","26,726,269,156.77","1,147,100,000.00","60,400.00","38,985,700,000.00","15,954,000,000.00","3,068,140,008.00","65,191,100,000.00","49,126,500,000.00","10,140,800,000.00",22.00,"5,252,345,950,571.46","138,818,542.00","51,656,500,864,631.46","34,559,985,418,528.91","3,580,814,046,075.57","59,293,915,820,799.34","72,742,547,231,696.05","21,064,699,384,324.01","17,102.00","512,336,028,056.76","28,583,144.00","14,719,262,499,973.70","7,540,458,479,487.50","1,451,938,537,430.88","18,882,661,702,751.00","19,112,391,499,318.30","4,391,251,531,038.75","1,386.00",0.10,0.21,0.28,0.22,0.41,0.32,0.26,0.21,0.08
211,BMU,USA,2019,"-370,913,936.94","5,942,394,792.47","5,261,472,357.00","123,442.00","76,520,956,341.00","53,230,845,962.00","6,270,485,742.84","87,178,568,755.00","90,819,277,063.00","14,298,320,722.00",22.00,"5,252,345,950,571.46","138,818,542.00","51,656,500,864,631.46","34,559,985,418,528.91","3,580,814,046,075.57","59,293,915,820,799.34","72,742,547,231,696.05","21,064,699,384,324.01","17,102.00","512,336,028,056.76","28,583,144.00","14,719,262,499,973.70","7,540,458,479,487.50","1,451,938,537,430.88","18,882,661,702,751.00","19,112,391,499,318.30","4,391,251,531,038.75","1,386.00",0.10,0.21,0.28,0.22,0.41,0.32,0.26,0.21,0.08
250,BRA,USA,2019,"-1,892,674,949.09","11,571,295,651.74","6,652,364,941.00","122,779.00","60,343,739,039.00","22,248,121,267.00","6,236,807,318.58","38,439,013,015.00","78,134,721,511.00","17,790,982,472.00",4.00,"5,2

### Step 4.3. Keep only the shares for the partners, and drop duplicates (effectively only keep one value for each iso_partner)

In [25]:
# Keep iso_partner and share_reported_total_profit_loss_by_partner	share_reported_total_n_employees_by_partner	share_reported_total_unrelated_party_revenues_by_partner	share_reported_total_tangible_assets_except_cash_by_partner	share_reported_total_payroll_by_partner	share_reported_total_stated_capital_by_partner	share_reported_total_total_revenues_by_partner	share_reported_total_related_party_revenues_by_partner	share_reported_total_holding_or_managing_ip_by_partner
shares_reported_2019 = final_misalignment_2019[['iso_partner', 'share_reported_total_profit_loss_by_partner', 'share_reported_total_n_employees_by_partner', 'share_reported_total_unrelated_party_revenues_by_partner', 'share_reported_total_tangible_assets_except_cash_by_partner', 'share_reported_total_payroll_by_partner', 'share_reported_total_stated_capital_by_partner', 'share_reported_total_total_revenues_by_partner', 'share_reported_total_related_party_revenues_by_partner', 'share_reported_total_holding_or_managing_ip_by_partner']]

# Drop duplicates
shares_reported_2019 = shares_reported_2019.drop_duplicates()

shares_reported_2019[shares_reported_2019['iso_partner'] == 'USA']

,iso_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
15,USA,0.10,0.21,0.28,0.22,0.41,0.32,0.26,0.21,0.08


### Step 4.4. Bring back the excluded countries

- The key here is the third command, where we sum by iso_parent. We basically assume that all countries report for the rest of the world, without caring about continents. **This could be improved and changed**. 
- Once that sum is done, we combine all potential iso_combinations for 2016 (created in Step 1), and we keep all combinations for the "excluded countries".
- Note in the test view, that no matter who is the iso_partner, the number in the variables will be the same because we have aggregated them. The next cells will now create the right shares.


In [26]:
excluded_2019 = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep if year == 2019 and iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_2019 = excluded_2019[excluded_2019['year'] == 2019]
excluded_2019 = excluded_2019[excluded_2019['iso_parent'].isin(['AUT', 'CZE', 'FIN', 'GRC', 'HUN', 'IMN', 'IRL', 'KOR', 'MAC', 'MUS', 'NZL', 'POL', 'SWE', 'GBR'])]

# Sum by iso_parent: 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected'
excluded_2019 = excluded_2019.groupby('iso_parent').agg({'n_employees': 'sum', 'unrelated_party_revenues': 'sum', 'tangible_assets_except_cash': 'sum', 'payroll': 'sum', 'stated_capital': 'sum', 'total_revenues': 'sum', 'related_party_revenues': 'sum', 'holding_or_managing_ip': 'sum', 'profit_loss_before_income_tax_corrected': 'sum'}).reset_index()

# Merge iso_combinations_2019 with excluded_2019. 
excluded_jurisdictions_2019 = pd.merge(iso_combinations_2019, excluded_2019, on='iso_parent', how='left')

# Drop year_y
excluded_jurisdictions_2019 = excluded_jurisdictions_2019.drop(columns=['year_y'])
# Rename year_x to year
excluded_jurisdictions_2019 = excluded_jurisdictions_2019.rename(columns={'year_x': 'year'})

# Keep if iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_jurisdictions_2019 = excluded_jurisdictions_2019[excluded_jurisdictions_2019['iso_parent'].isin(['AUT', 'CZE', 'FIN', 'GRC', 'HUN', 'IMN', 'IRL', 'KOR', 'MAC', 'MUS', 'NZL', 'POL', 'SWE', 'GBR'])]

excluded_jurisdictions_2019[excluded_jurisdictions_2019['iso_partner'] == 'USA']


,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
616,AUT,2019,USA,"2,008,493.00","626,699,874,863.00","376,167,584,865.00","14,217,046,223.71","234,583,578,395.00","793,870,310,447.00","167,156,841,265.00","1,196.00","36,984,207,173.65"
2506,CZE,2019,USA,"475,313.00","150,183,564,317.00","46,574,735,398.00","2,136,644,945.28","30,434,844,214.00","268,518,186,031.00","118,419,219,859.00",0.00,"5,221,484,424.00"
3346,FIN,2019,USA,"1,040,283.00","389,022,346,262.00","152,296,740,600.00","6,970,656,939.12","456,209,859,738.00","535,647,116,164.00","146,623,327,023.00",211.00,"38,129,397,314.00"
3766,GBR,2019,USA,"13,630,674.00","4,694,181,530,169.00","2,922,287,144,908.00","121,952,200,891.46","12,595,160,253,859.00","6,329,438,247,967.00","1,633,919,717,782.00","2,459.00","424,695,866,422.00"
3976,GRC,2019,USA,"224,505.00","81,347,343,203.66","53,804,213,410.14","2,587,691,619.01","123,919,093,719.92","97,592,559,665.80","16,245,216,458.04",176.00,"-913,518,434.12"
4396,HUN,2019,USA,"106,943.00","38,156,447,266.00","16,163,035,372.00","596,581,611.66","14,247,407,989.00","50,004,235,099.00","11,847,787,833.00",0.00,"4,244,515,221.00"
4816,IMN,2019,USA,"104,400.00","25,990,618,676.00","12,466,257,969.00","38,535,428.43","78,646,673,078.00","40,923,664,328.00","14,933,045,651.00",44.00,"8,159,443,951.00"
5236,IRL,2019,USA,"1,742,776.00","411,414,367,240.00","196,933,889,617.00","5,427,879,940.58","3,714,190,000,000.00","626,742,187,641.00","215,333,953,159.00",485.00,"-1,426,697,053.00"
5866,KOR,2019,USA,"5,344,161.00","2,526,385,408,263.00","1,741,605,391,577.00","69,502,551,975.53","838,731,311,848.00","3,617,231,834,032.00","1,092,030,052,295.00","1,009.00","131,240,234,946.00"
6706,MAC,2019,USA,"31,452.00","6,609,971,676.00","17,690,030,788.00","651,276,227.14","2,620,744,511.00","7,104,986,177.00","495,014,499.00",0.00,"1,126,361,990.00"


### Step 4.5. Merge with the shares reported, and multiply the number

In [27]:
# Merge with share_reported_2019
excluded_jurisdictions_share_reported_2019 = pd.merge(excluded_jurisdictions_2019, shares_reported_2019, on='iso_partner', how='left')

# Multiply 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected' by share_reported
excluded_jurisdictions_share_reported_2019['n_employees'] = excluded_jurisdictions_share_reported_2019['n_employees'] * excluded_jurisdictions_share_reported_2019['share_reported_total_n_employees_by_partner']
excluded_jurisdictions_share_reported_2019['unrelated_party_revenues'] = excluded_jurisdictions_share_reported_2019['unrelated_party_revenues'] * excluded_jurisdictions_share_reported_2019['share_reported_total_unrelated_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2019['tangible_assets_except_cash'] = excluded_jurisdictions_share_reported_2019['tangible_assets_except_cash'] * excluded_jurisdictions_share_reported_2019['share_reported_total_tangible_assets_except_cash_by_partner']
excluded_jurisdictions_share_reported_2019['payroll'] = excluded_jurisdictions_share_reported_2019['payroll'] * excluded_jurisdictions_share_reported_2019['share_reported_total_payroll_by_partner']
excluded_jurisdictions_share_reported_2019['stated_capital'] = excluded_jurisdictions_share_reported_2019['stated_capital'] * excluded_jurisdictions_share_reported_2019['share_reported_total_stated_capital_by_partner']
excluded_jurisdictions_share_reported_2019['total_revenues'] = excluded_jurisdictions_share_reported_2019['total_revenues'] * excluded_jurisdictions_share_reported_2019['share_reported_total_total_revenues_by_partner']
excluded_jurisdictions_share_reported_2019['related_party_revenues'] = excluded_jurisdictions_share_reported_2019['related_party_revenues'] * excluded_jurisdictions_share_reported_2019['share_reported_total_related_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2019['holding_or_managing_ip'] = excluded_jurisdictions_share_reported_2019['holding_or_managing_ip'] * excluded_jurisdictions_share_reported_2019['share_reported_total_holding_or_managing_ip_by_partner']
excluded_jurisdictions_share_reported_2019['profit_loss_before_income_tax_corrected'] = excluded_jurisdictions_share_reported_2019['profit_loss_before_income_tax_corrected'] * excluded_jurisdictions_share_reported_2019['share_reported_total_profit_loss_by_partner']

# Drop share_reported columns
excluded_jurisdictions_dataset_2019 = excluded_jurisdictions_share_reported_2019.drop(columns=[
    'share_reported_total_profit_loss_by_partner',
    'share_reported_total_n_employees_by_partner',
    'share_reported_total_unrelated_party_revenues_by_partner',
    'share_reported_total_tangible_assets_except_cash_by_partner',
    'share_reported_total_payroll_by_partner',
    'share_reported_total_stated_capital_by_partner',
    'share_reported_total_total_revenues_by_partner',
    'share_reported_total_related_party_revenues_by_partner',
    'share_reported_total_holding_or_managing_ip_by_partner'
])

excluded_jurisdictions_dataset_2019[excluded_jurisdictions_dataset_2019['iso_partner'] == 'USA']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
196,AUT,2019,USA,"413,554.59","178,575,006,289.77","82,073,994,553.33","5,764,688,429.79","74,705,174,899.24","208,581,645,135.70","34,846,342,771.68",96.93,"3,607,595,916.66"
406,CZE,2019,USA,"97,868.34","42,794,058,237.83","10,161,892,553.16","866,360,860.11","9,692,240,077.48","70,550,522,237.90","24,686,256,899.41",0.00,"509,325,664.29"
616,FIN,2019,USA,"214,197.31","110,849,978,940.53","33,228,811,735.55","2,826,442,621.95","145,283,986,183.80","140,736,031,101.57","30,565,824,725.46",17.10,"3,719,302,604.19"
826,GBR,2019,USA,"2,806,595.66","1,337,583,634,364.87","637,598,211,181.68","49,448,839,822.58","4,011,038,010,785.72","1,662,997,879,089.86","340,614,994,375.10",199.29,"41,426,630,192.00"
1036,GRC,2019,USA,"46,226.24","23,179,520,065.21","11,739,253,715.75","1,049,250,013.06","39,463,110,048.20","25,641,488,767.40","3,386,558,257.58",14.26,"-89,108,449.92"
1246,HUN,2019,USA,"22,019.88","10,872,489,502.27","3,526,526,288.98","241,900,255.51","4,537,210,630.69","13,138,122,793.42","2,469,848,513.47",0.00,"414,027,958.14"
1456,IMN,2019,USA,"21,496.27","7,405,897,271.89","2,719,946,188.39","15,625,238.53","25,045,715,082.59","10,752,291,801.55","3,113,016,634.21",3.57,"795,906,656.63"
1666,IRL,2019,USA,"358,842.68","117,230,473,731.38","42,967,952,673.52","2,200,881,688.04","1,182,816,015,756.19","164,670,368,514.60","44,889,582,055.83",39.31,"-139,166,061.84"
1876,KOR,2019,USA,"1,100,378.39","719,880,932,271.73","379,991,570,708.16","28,181,701,804.28","267,101,259,916.70","950,392,220,052.66","227,649,991,656.15",81.77,"12,801,727,328.37"
2086,MAC,2019,USA,"6,476.06","1,883,478,489.40","3,859,693,256.30","264,077,676.34","834,598,817.19","1,866,765,498.05","103,193,173.42",0.00,"109,870,110.15"


## Step 5. Calculate Misalignment with all countries reporting in the CBCR

### Step 5.1. Concatenate the two samples

- Concatenate the cbcr_sample (without the "bad reporters") and the sample with the bad reporters.
- Ensure all required columns are included: 'iso_parent', 'year', 'iso_partner', 'n_employees', 'unrelated_party_revenues', 'tangible_assets_except_cash', 'payroll', 'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip', 'profit_loss_before_income_tax_corrected'.

In [28]:
final_misalignment_2019 = cbcr_sample[cbcr_sample['year'] == 2019].copy()

# Concatenate excluded_jurisdictions_dataset_2016
final_misalignment_2019 = pd.concat([final_misalignment_2019, excluded_jurisdictions_dataset_2019])

# Save the final misalignment dataset
#final_misalignment_2019.to_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2019.csv', index=False) 

final_misalignment_2019[final_misalignment_2019['iso_partner'] == 'ZAF']

,iso_parent,parent_jurisdiction,iso_partner,partner_jurisdiction,year,unrelated_party_revenues,profit_loss_before_income_tax,adjusted_profit_loss_before_income_tax,income_tax_paid_on_cash_basis,income_tax_accrued_current_year,n_employees,tangible_assets_except_cash,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,n_cbcr,n_cbcr_groups,n_entities,profit_loss_before_income_tax_corrected,ln_profit_loss_before_income_tax_corrected,ln_unrelated_party_revenues,ln_n_employees,ln_tangible_assets_except_cash,ln_stated_capital,ln_total_revenues,ln_related_party_revenues,ln_holding_or_managing_ip,etr_domestic,etr_domestic_corrected,etr_foreign,etr_foreign_corrected,etr_average,etr_average_corrected,cit,gdp_current_usd,population,gdp,wage_monthly,payroll,ln_wage_monthly,ln_gdp_current_usd,ln_population,gvt_health_expenditure,ln_gvt_health_expenditure,tax_revenue_pct_gdp,tax_revenue_current_usd,cthi_2021_share,cthi_2021_score,region_tjn,ukt,gbr_oct,nld_oct,oecd_oct,oecd,eu
262,ARG,Argentina,ZAF,South Africa,2019,"28,759.50","-1,380,866.27",NaN,"-1,828.06",457.28,1.00,0.00,"-1,367,475.58","28,759.50",0.00,0.00,2.00,2.00,2.00,"-1,380,866.27",0.00,10.27,0.69,0.00,0.00,10.27,0.00,0.00,0.10,0.13,0.19,0.20,0.12,0.15,0.28,"389,330,032,224.27","58,087,055.00",NaN,263.00,"3,156.05",5.58,26.69,17.88,"18,688,090,576.76",23.65,24.87,"96,816,043,586.61",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
749,AUS,Australia,ZAF,South Africa,2019,"2,674,200,220.00","-520,275,002.00",NaN,"78,157,684.00","25,102,543.00","13,554.00","1,740,767,202.00","4,017,181,536.00","3,957,608,330.00","1,283,408,110.00",5.00,23.00,23.00,129.00,"-520,275,002.00",0.00,21.71,9.51,21.28,22.11,22.10,20.97,1.79,0.10,0.13,0.19,0.20,0.12,0.15,0.28,"389,330,032,224.27","58,087,055.00",NaN,263.00,"42,777,074.59",5.58,26.69,17.88,"18,688,090,576.76",23.65,24.87,"96,816,043,586.61",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
1641,BMU,Bermuda,ZAF,South Africa,2019,"354,152,194.40","19,236,151.23",NaN,"1,274,595.18","1,856,154.40",984.00,"84,416,478.32","72,346,232.54","397,311,899.70","43,159,705.32",NaN,21.00,21.00,40.00,"19,236,151.23",16.77,19.69,6.89,18.25,18.10,19.80,17.58,NaN,0.10,0.13,0.19,0.20,0.12,0.15,0.28,"389,330,032,224.27","58,087,055.00",NaN,263.00,"3,105,551.23",5.58,26.69,17.88,"18,688,090,576.76",23.65,24.87,"96,816,043,586.61",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
1890,BRA,Brazil,ZAF,South Africa,2019,"394,081,253.00","-67,666,834.00",NaN,"10,567,377.00","10,354,735.00","1,614.00","186,364,247.00","58,314,195,898.00","490,925,512.00","96,844,258.00",0.00,11.00,11.00,29.00,"-67,666,834.00",0.00,19.79,7.39,19.04,24.79,20.01,18.39,0.00,0.10,0.13,0.19,0.20,0.12,0.15,0.28,"389,330,032,224.27","58,087,055.00",NaN,263.00,"5,093,861.47",5.58,26.69,17.88,"18,688,090,576.76",23.65,24.87,"96,816,043,586.61",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
2711,CHE,Switzerland,ZAF,South Africa,2019,"9,726,972,493.00","-288,453,660.00",NaN,"120,783,992.00","-4,941,754.00","3,452.00","7,341,447,985.00","3,298,539,497.00","11,832,818,812.00","2,105,847,317.00",3.00,57.00,57.00,253.00,"-288,453,660.00",0.00,23.00,8.15,22.72,21.92,23.19,21.47,1.39,0.10,0.13,0.19,0.20,0.12,0.15,0.28,"389,330,032,224.27","58,087,055.00",NaN,263.00,"10,894,677.70",5.58,26.69,17.88,"18,688,090,576.76",23.65,24.87,"96,816,043,586.61",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
3561,CHN,China (People’s Republic of),ZAF,South Africa,2019,"5,667,441,916.00","428,701,450.30",NaN,"105,542,873.80","126,153,812.70","11,319.00","4,362,137,075.00","1,843,420,497.00","7,799,073,233.00","2,137,949,950.00",7.00,83.00,83.00,139.00,"428,701,450.30",19.88,22.46,9.33,22.20,21.33,22.78,21.48,2.08,0.10,0.13,0.19,0.20,0.12,0.15,0.28,"389,330,032,224.27","58,087,055.00",NaN,263.00,"35,723,307.31",5.58,26.69,17.88,"18,688,090,576.76",23.65,24.87,"96,816,043,586.61",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
4131,CYM,Cayman Islands,ZAF,South Africa,2019,"93,624,369.00","-1,280,525.

### Step 5.2. Perform the misalignment estimates, and all the remaining calculations needed

In [29]:
# Initialize a list to store the aggregate results
results_sample = []

# Start the estimates
misalignment_final_estimates_2019 = final_misalignment_2019[final_misalignment_2019['year'] == 2019].copy()
misalignment_final_estimates_2019 = calculate_misalignment(misalignment_final_estimates_2019, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Perform the groupby operation on 'iso_partner'
country_results_2019 = misalignment_final_estimates_2019.groupby(['iso_partner']).agg(
    negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    theoretical_profit=('theoretical_profit', 'sum'),
    reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
).reset_index()

# Convert results to millions
country_results_2019['negative_misalignment'] = -country_results_2019['negative_misalignment'] / 1e6
country_results_2019['positive_misalignment'] = country_results_2019['positive_misalignment'] / 1e6
country_results_2019['theoretical_profit'] = country_results_2019['theoretical_profit'] / 1e6
country_results_2019['reported_profit'] = country_results_2019['reported_profit'] / 1e6

# Merge the unique columns back into the result
country_results_2019 = country_results_2019.merge(unique_columns, on='iso_partner', how='left')

# Calculate other relevant variables
country_results_2019['tax_revenue_loss'] = country_results_2019['negative_misalignment'] * country_results_2019['cit']
country_results_2019['tax_revenue_gain'] = country_results_2019['positive_misalignment'] * country_results_2019['etr_average_corrected']

country_results_2019['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
    country_results_2019['gvt_health_expenditure'] == 0, 
    np.nan, 
    country_results_2019['tax_revenue_loss'] / (country_results_2019['gvt_health_expenditure'] / 1e6)
)
    
country_results_2019['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
    country_results_2019['tax_revenue_current_usd'] == 0, 
    np.nan, 
    country_results_2019['tax_revenue_loss'] / (country_results_2019['tax_revenue_current_usd'] / 1e6)
)

# Calculate totals
total_positive_misalignment = country_results_2019['positive_misalignment'].sum()
total_negative_misalignment = country_results_2019['negative_misalignment'].sum()
total_profits = country_results_2019['reported_profit'].sum()
misaligned_of_total_profits = total_positive_misalignment / total_profits
total_tax_revenue_loss = country_results_2019['tax_revenue_loss'].sum()
total_tax_revenue_gain = country_results_2019['tax_revenue_gain'].sum()
average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_2019['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_2019['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

print(f"Year {2019}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
        f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

# Calculate countries' fractions of totals
country_results_2019['tax_revenue_loss_caused_pct_of_total'] = country_results_2019['positive_misalignment'] / total_positive_misalignment
country_results_2019['tax_revenue_loss_caused_usd'] = country_results_2019['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
country_results_2019['tax_revenue_loss_suffered_pct_of_total'] = country_results_2019['tax_revenue_loss'] / total_tax_revenue_loss

country_results_2019 = country_results_2019.sort_values(by='iso_partner')
country_results_2019.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2019.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
# Append aggregate results to the list
results_sample.append({
    'year': 2019,
    'total_positive_misalignment': total_positive_misalignment,
    'total_negative_misalignment': total_negative_misalignment,
    'total_profits': total_profits,
    'misaligned_of_total_profits': misaligned_of_total_profits,
    'total_tax_revenue_loss': total_tax_revenue_loss,
    'total_tax_revenue_gain': total_tax_revenue_gain,
    'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
    'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
})

# Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2019.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE


Year 2019: Positive Misalignment: 1162186.8677575332, Negative Misalignment: 1162186.8677575332, Shifted of total profits: 0.19228309544476496, Total tax revenue loss: 306039.3340631637, Total tax revenue gain: 107502.66922670024


/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_6423/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


## Step 6. Checking the datasets

### Step 6.1. Checking the countries


In [30]:
# Open the CSV file f'{output_tables}/Scaling_Mario/SOTJ_sample_2016.csv'
sotj_2019_countries = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2019.csv')
sotj_2019_countries

,iso_partner,negative_misalignment,positive_misalignment,theoretical_profit,reported_profit,partner_jurisdiction,etr_average_corrected,cit,tax_revenue_current_usd,gvt_health_expenditure,region_tjn,ukt,oecd,oecd_oct,nld_oct,tax_revenue_loss,tax_revenue_gain,tax_revenue_loss_pct_of_gvt_health_expenditure,tax_revenue_loss_pct_of_total_tax_revenues,tax_revenue_loss_caused_pct_of_total,tax_revenue_loss_caused_usd,tax_revenue_loss_suffered_pct_of_total
0,ABW,0.12,0.00,59.89,84.36,Aruba,0.20,0.25,NaN,NaN,Caribbean/American isl.,0.00,0.00,1.00,1.00,0.03,0.00,NaN,NaN,0.00,0.00,0.00
1,AFG,14.27,4.87,15.48,1.39,Afghanistan,0.07,0.20,NaN,"93,965,898.86",Asia,0.00,0.00,0.00,0.00,2.85,0.32,0.03,NaN,0.00,1.28,0.00
2,AGO,32.96,335.11,336.75,"2,878.77",Angola,0.36,0.30,"7,153,877,964.00","949,590,173.24",Africa,0.00,0.00,0.00,0.00,9.89,121.41,0.01,0.00,0.00,88.25,0.00
3,AIA,3.13,4.54,4.54,5.91,Anguilla,0.00,0.00,NaN,NaN,Caribbean/American isl.,1.00,0.00,1.00,0.00,0.00,0.00,NaN,NaN,0.00,1.19,0.00
4,ALB,55.61,25.98,176.55,121.28,Albania,0.08,0.15,"2,794,565,426.68","451,085,348.78",Europe,0.00,0.00,0.00,0.00,8.34,1.97,0.02,0.00,0.00,6.84,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205,XKV,1.84,0.20,4.06,2.03,Kosovo,0.08,0.10,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.18,0.02,NaN,NaN,0.00,0.05,0.00
206,YEM,228.45,0.28,234.21,2.67,Yemen,0.45,0.20,NaN,NaN,Asia,0.00,0.00,0.00,0.00,45.69,0.12,NaN,NaN,0.00,0.07,0.00
207,ZAF,"4,619.19",616.20,"18,011.49","12,696.00",South Africa,0.15,0.28,"96,816,043,586.61","18,688,090,576.76",Africa,0.00,0.00,0.00,0.00,"1,293.37",89.95,0.07,0.01,0.00,162.27,0.00
208,ZMB,"1,377.65",0.62,"1,283.40",-357.01,Zambia,0.16,0.35,"3,887,331,659.63","526,383,743.13",Africa,0.00,0.00,0.00,0.00,482.18,0.10,0.92,0.12,0.00,0.16,0.00


### Step 6.2. Checking the aggregate results

In [31]:
sotj_2019_aggregate = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2019.csv')
sotj_2019_aggregate

,year,total_positive_misalignment,total_negative_misalignment,total_profits,misaligned_of_total_profits,total_tax_revenue_loss,total_tax_revenue_gain,average_tax_revenue_loss_pct_of_gvt_health_expenditure,average_tax_revenue_loss_pct_of_total_tax_revenues
0,2019,"1,162,186.87","1,162,186.87","6,044,144.78",0.19,"306,039.33","107,502.67",0.40,0.03
